# Theorem 1 — Vocabulary reduction under drift with trigram simulations

This notebook studies **vocabulary reduction under recursive drift** in word-trigram models.

It is designed for the theorem-1 experiments discussed in the paper:
- literary corpora such as **Arthur Conan Doyle**, **Jane Austen**, and **Charles Darwin**;
- **fixed-size** environments where an `alpha` fraction of the corpus is replaced by machine-generated text at each generation;
- **growing-corpus** environments where the original corpus stays in place and additional machine-generated text accumulates over time.

The notebook measures, generation by generation:
- total token count;
- corpus vocabulary size;
- **head vocabulary size** (distinct word types appearing in the left-hand side/context positions of observed trigrams);
- number of distinct trigram types;
- active vocabulary above the baseline threshold `1 / M0`, where `M0` is the original corpus size in tokens.

It also writes a SQLite database and generation-wise CSV files so the appendix can later support analytic work.


## Repository assumptions

The notebook is intended to live at:

```text
Drift_and_selection/GitHub/notebooks/active/17_theorem1_vocabulary_drift_trigrams.ipynb
```

and expects the helper module at either:

```text
Drift_and_selection/GitHub/src/drift_selection/theorem1_vocab_drift.py
```

or in the same folder as the notebook.

Corpora are expected under:

```text
Drift_and_selection/GitHub/corpora/raw/<author>/
Drift_and_selection/GitHub/corpora/cleaned/<author>/
```

Outputs go to:

```text
Drift_and_selection/GitHub/data/outputs/theorem1_vocabulary_drift/
Drift_and_selection/GitHub/data/databases/theorem1_vocabulary_drift.sqlite
```


In [ ]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path


def _import_helper_module():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    helper_paths = []
    for cand in candidates:
        helper_paths.append(cand / "theorem1_vocab_drift.py")
        helper_paths.append(cand / "src" / "drift_selection" / "theorem1_vocab_drift.py")
        helper_paths.append(cand / "GitHub" / "src" / "drift_selection" / "theorem1_vocab_drift.py")
    for path in helper_paths:
        if path.exists():
            spec = importlib.util.spec_from_file_location("theorem1_vocab_drift", path)
            module = importlib.util.module_from_spec(spec)
            assert spec and spec.loader
            sys.modules["theorem1_vocab_drift"] = module
            spec.loader.exec_module(module)
            return module, path
    raise FileNotFoundError(
        "Could not locate theorem1_vocab_drift.py. Put it next to the notebook or under src/drift_selection/."
    )

mod, helper_path = _import_helper_module()
print(f"Imported helper module from: {helper_path}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8.5, 5.5)
plt.rcParams["axes.grid"] = True

paths = mod.default_paths()
paths


## 1. Corpus preparation

The helper module knows about three author bundles:
- `conan_doyle`
- `jane_austen`
- `charles_darwin`

If a cleaned combined corpus already exists, it is reused.
If not, the helper tries to download public-domain texts from Project Gutenberg, strip headers/footers, and build a combined cleaned text file for the author.

For Conan Doyle, if you already have a cleaned text file, simply place it at:

```text
Drift_and_selection/GitHub/corpora/cleaned/conan_doyle/conan_doyle_combined_clean.txt
```

before running the download cell.


In [ ]:
# Toggle downloads author by author. Conan Doyle may already be present locally.
BUILD_AUTHORS = ["conan_doyle"]  # add "jane_austen" and "charles_darwin" after Conan pilot succeeds

clean_paths = {}
for author_key in BUILD_AUTHORS:
    clean_paths[author_key] = mod.build_or_download_author_corpus(author_key, force_download_missing=True)
    print(author_key, "->", clean_paths[author_key])


In [ ]:
# Inspect a corpus quickly.
CORPUS_NAME = "conan_doyle"   # change to jane_austen or charles_darwin
LOWERCASE_TOKENS = True

sentences = mod.load_clean_corpus(CORPUS_NAME, lowercase=LOWERCASE_TOKENS, progress_bar=True)
metrics_df, top_words_df, top_trigrams_df = mod.summarize_corpus(sentences, top_k=20)
print(f"Number of sentences: {len(sentences):,}")
metrics_df


In [ ]:
top_words_df.head(20)


In [ ]:
top_trigrams_df.head(20)


## 2. Sanity check: build a trigram model and generate sample sentences

This is just to verify that the tokenization and trigram model look sensible before running the full recursive simulations.


In [ ]:
import random

rng = random.Random(12345)
model = mod.build_trigram_model(sentences)
for i in range(5):
    sample = mod.sample_sentence(model, rng=rng, max_tokens=40)
    print(f"[{i+1}]", " ".join(sample))


## 3. Fixed-size corpus simulations

Interpretation: each generation keeps a fraction `(1 - alpha)` of the **original anchor corpus** and replaces a fraction `alpha` with newly generated trigram text.

This matches the theorem-1 mixed-environment intuition: the public corpus size stays fixed, but more of it becomes recursively generated over time.


In [ ]:
ALPHAS = [0.1, 0.25, 0.5, 0.75, 1.0]
GENERATIONS = 10
REPLICATE_SEEDS = [12345, 22345, 32345]
SAVE_FULL_TRIGRAM_TABLES = False   # turn on for appendix-grade runs if storage is acceptable

fixed_frames = []
for rep_idx, base_seed in enumerate(REPLICATE_SEEDS, start=1):
    run_name = f"theorem1_{CORPUS_NAME}_fixedsize_pilot_rep{rep_idx}"
    print(f"[pilot] fixed-size run: {run_name}")
    df_rep = mod.run_alpha_grid(
        anchor_sentences=sentences,
        corpus_name=CORPUS_NAME,
        alphas=ALPHAS,
        generations=GENERATIONS,
        mode="fixed_size_mix",
        base_seed=base_seed,
        save_full_trigram_tables=SAVE_FULL_TRIGRAM_TABLES,
        paths=paths,
        run_name=run_name,
        resume=True,
        force_rebuild=False,
        progress_bar=True,
    )
    df_rep["replicate"] = rep_idx
    fixed_frames.append(df_rep)

fixed_df = pd.concat(fixed_frames, ignore_index=True)
fixed_df.head()

In [ ]:
fig = mod.plot_metric_grid(
    fixed_df,
    metric="vocab_size",
    title=f"Theorem 1 fixed-size drift: vocabulary size ({CORPUS_NAME})",
    out_path_base=paths["figures"] / f"{CORPUS_NAME}_fixed_size_vocab_size",
)
plt.show()


In [ ]:
fig = mod.plot_metric_grid(
    fixed_df,
    metric="head_vocab_size",
    title=f"Theorem 1 fixed-size drift: trigram-head vocabulary ({CORPUS_NAME})",
    out_path_base=paths["figures"] / f"{CORPUS_NAME}_fixed_size_head_vocab",
)
plt.show()


In [ ]:
fig = mod.plot_metric_grid(
    fixed_df,
    metric="trigram_type_count",
    title=f"Theorem 1 fixed-size drift: trigram type count ({CORPUS_NAME})",
    out_path_base=paths["figures"] / f"{CORPUS_NAME}_fixed_size_trigram_types",
)
plt.show()


## 4. Growing-corpus simulations

Interpretation: the original corpus stays in place and each generation adds `alpha * M0` machine-generated tokens, where `M0` is the original token count.

In this regime, raw vocabulary size may grow or remain stable simply because the total corpus grows. For this reason we also track **active vocabulary** above the fixed threshold `1 / M0`.


In [ ]:
GROWING_ALPHAS = [0.05, 0.1, 0.25, 0.5]
GROWING_GENERATIONS = 10

growing_frames = []
for rep_idx, base_seed in enumerate(REPLICATE_SEEDS, start=1):
    run_name = f"theorem1_{CORPUS_NAME}_growing_pilot_rep{rep_idx}"
    print(f"[pilot] growing run: {run_name}")
    df_rep = mod.run_alpha_grid(
        anchor_sentences=sentences,
        corpus_name=CORPUS_NAME,
        alphas=GROWING_ALPHAS,
        generations=GROWING_GENERATIONS,
        mode="growing_corpus",
        base_seed=base_seed + 1000,
        save_full_trigram_tables=SAVE_FULL_TRIGRAM_TABLES,
        paths=paths,
        run_name=run_name,
        resume=True,
        force_rebuild=False,
        progress_bar=True,
    )
    df_rep["replicate"] = rep_idx
    growing_frames.append(df_rep)

growing_df = pd.concat(growing_frames, ignore_index=True)
growing_df.head()

In [ ]:
fig = mod.plot_metric_grid(
    growing_df,
    metric="vocab_size",
    title=f"Theorem 1 growing corpus: raw vocabulary size ({CORPUS_NAME})",
    out_path_base=paths["figures"] / f"{CORPUS_NAME}_growing_vocab_size",
)
plt.show()


In [ ]:
if "active_vocab_gt_1_over_M0" in growing_df.columns:
    fig = mod.plot_metric_grid(
        growing_df,
        metric="active_vocab_gt_1_over_M0",
        title=f"Theorem 1 growing corpus: active vocabulary > 1/M0 ({CORPUS_NAME})",
        out_path_base=paths["figures"] / f"{CORPUS_NAME}_growing_active_vocab",
    )
    plt.show()


## 5. Inspect the database and exported counts

Each run is registered in:

```text
GitHub/data/databases/theorem1_vocabulary_drift.sqlite
```

and each generation also exports word counts, bigram-context counts, and trigram counts (full or top-5000 depending on configuration).


In [ ]:
import sqlite3

with sqlite3.connect(paths["db"]) as conn:
    runs_df = pd.read_sql_query("SELECT * FROM runs ORDER BY run_id DESC LIMIT 10", conn)
    metrics_preview_df = pd.read_sql_query(
        "SELECT * FROM generation_metrics ORDER BY run_id DESC, generation ASC LIMIT 20",
        conn,
    )

runs_df


In [ ]:
metrics_preview_df


In [ ]:
run_dirs = sorted((paths["outputs"] / "runs").glob("*"))
run_dirs[-5:] if run_dirs else []


## 6. Suggested experiment sweeps for the appendix

Once the pipeline is working, the appendix can systematically sweep:

- **corpus**: Conan Doyle, Austen, Darwin
- **mode**: fixed-size mix vs growing corpus
- **alpha**: e.g. `0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0`
- **generation count**: e.g. `10, 20, 50`
- **normalization**: lowercase on/off if needed
- **storage mode**: top trigram counts vs full trigram tables

Natural theorem-1 figures:
- vocabulary size vs generation
- head vocabulary size vs generation
- active vocabulary size vs generation
- trigram type count vs generation
- entropy vs generation

Natural appendix tables:
- final-generation vocabulary retention for each corpus and alpha
- decay rates under fixed-size vs growing-corpus regimes
- top disappearing words / phrases by generation


## 7. Instructions for Codex

If Codex is organizing the repository, it should:

1. place this notebook at:
   - `Drift_and_selection/GitHub/notebooks/active/17_theorem1_vocabulary_drift_trigrams.ipynb`
2. place the helper module at:
   - `Drift_and_selection/GitHub/src/drift_selection/theorem1_vocab_drift.py`
3. ensure corpora live under:
   - `GitHub/corpora/raw/<author>/`
   - `GitHub/corpora/cleaned/<author>/`
4. ensure outputs go to:
   - `GitHub/data/outputs/theorem1_vocabulary_drift/`
   - `GitHub/data/databases/theorem1_vocabulary_drift.sqlite`
5. keep the SQLite database plus per-generation CSV exports, so later appendix work can derive exact formulas or compare against analytic expectations.
